# Synthetic Data Comparison — Simulator Launcher

Run the steps in order. The simulator opens at **http://localhost:5000**.

1. **Step 1 — Install requirements** (run only the first time)
2. **Step 2 — Setup & options** (run every launch)
3. **Step 3 — Start the server** (opens your browser when ready)
4. **Step 4 — Stop the server**

## Step 1 — Install requirements (first time only)

Run this **once** the first time you use the simulator. On later launches you can **skip Step 1** and start from Step 2.

In [ ]:
import os, pathlib, subprocess, sys

try:
    NOTEBOOK_DIR = str(pathlib.Path(__vsc_ipynb_file__).parent)  # VS Code
except NameError:
    try:
        import ipynbname
        NOTEBOOK_DIR = str(ipynbname.path().parent)              # Jupyter/Spyder
    except Exception:
        NOTEBOOK_DIR = os.getcwd()                               # fallback

req_file = os.path.join(NOTEBOOK_DIR, "requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_file])

## Step 2 — Setup & options

Resolves the folders and sets the launch options. Run this **every time** you start the simulator.

In [ ]:
import os, pathlib

try:
    NOTEBOOK_DIR = str(pathlib.Path(__vsc_ipynb_file__).parent)  # VS Code
except NameError:
    try:
        import ipynbname
        NOTEBOOK_DIR = str(ipynbname.path().parent)              # Jupyter/Spyder
    except Exception:
        NOTEBOOK_DIR = os.getcwd()                               # fallback

SERVER_DIR = os.path.join(NOTEBOOK_DIR, "soft-sensor-based-adaptation-learning")

print("Notebook directory:", NOTEBOOK_DIR)
print("Server directory:  ", SERVER_DIR)

## Step 3 — Start the server

Starts the server and opens the simulator in your browser

In [ ]:
import threading, webbrowser, time, sys, socket, importlib, logging
from werkzeug.serving import make_server

# Hide the per-request access logs (errors still show)
logging.getLogger("werkzeug").setLevel(logging.ERROR)

sys.path.insert(0, NOTEBOOK_DIR)   # algorithm .py files
sys.path.insert(0, SERVER_DIR)     # server.py imports
os.chdir(SERVER_DIR)

# Stop a previous server if Step 3 is re-run without Step 4
if "httpd" in globals():
    try:
        httpd.shutdown()
    except Exception:
        pass

if "server" in sys.modules:
    server_mod = importlib.reload(sys.modules["server"])
else:
    import server as server_mod

httpd = make_server("127.0.0.1", 5000, server_mod.app, threaded=True)
server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()

def wait_for_server(host="127.0.0.1", port=5000, timeout=30):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.25)
    return False

if wait_for_server():
    webbrowser.open("http://localhost:5000")
    print("Keep this notebook open while using the simulator.")
else:
    print("Server did not start within 30s - open http://localhost:5000 manually once it is ready.")

## Step 4 — Stop the server

Run this to shut the server down. Re-run **Step 3** to start it again.

In [3]:
try:
    httpd.shutdown()
    server_thread.join(timeout=5)
    print("Server stopped.")
except NameError:
    print("No server is running.")

Server stopped.
